# De la medicion con el analizador de espectro a lo que ve la juntura

Este notebook es el paso siguiente a `filtros_expLT.ipynb`: ahi se armo y valido el modelo de analisis nodal de los filtros contra los valores de la tesis de Fran (Seccion 5.2). Ahora arranca la etapa de medicion real (item 3 del plan: *"Analisis de las etapas de filtrado ... con analizador de espectro"*).

## El problema

Lo que en realidad interesa es la senal **en la juntura**, adentro del criostato -- un punto imposible de medir directamente. Fran confirmo que se puede medir **toda la etapa a temperatura ambiente**: tanto la entrada del filtro (donde se conectan los instrumentos) como la salida (justo donde el cable entra al criostato), porque las dos estan en la caja de aluminio, sobre la mesa. Lo unico que sigue siendo inaccesible es la etapa criogenica, adentro del criostato.

## ¿Cómo se mide esto, en la práctica?

Ojo con una confusión común: **toda** medición de voltaje es una diferencia entre dos puntos, nunca hay "un solo punto" -- pero acá **no es una diferencia entre A y B**. Son dos mediciones separadas, cada una tomada contra la misma tierra (la malla del cable del analizador, conectada a la caja de aluminio):

- Medición 1: punta en **A** (entrada del filtro ambiente) vs. tierra → $V_A(f)$.
- Medición 2: punta en **B** (salida del filtro ambiente, justo antes de que el cable entre al criostato) vs. tierra → $V_B(f)$.

Es como medir la altura de dos pisos de un edificio con una cinta métrica desde la vereda cada vez, no estirando la cinta de un piso al otro directamente: cada altura es una diferencia respecto del mismo nivel de referencia, y por eso después se pueden comparar entre sí.

## La estrategia

En vez de confiar en el modelo simulado para **toda** la cadena (como se planteo en un primer intento), conviene medir todo lo que se pueda y dejarle al modelo solo el tramo que de verdad no se puede tocar:

$$\underbrace{V_{entrada}(f)}_{\text{medido}} \ \xrightarrow{\ H_{ambiente}(f)\ \textbf{medido}\ }\ \underbrace{V_{salida\ filtro\ ambiente}(f)}_{\text{medido}} \ \xrightarrow{\ H_{cry}(f)\ \textbf{simulado}\ }\ \underbrace{V_{juntura}(f)}_{\text{estimado}}$$

- $H_{ambiente}(f) = V_{salida}/V_{entrada}$: se mide directamente con el analizador en los dos extremos del filtro ambiente. No hace falta confiar en los valores nominales de $R_{ta}$/$C_{ta}$ -- se esta midiendo el filtro real, con sus tolerancias reales.
- $H_{cry}(f)$: no se puede medir (esta adentro del criostato), asi que sigue viniendo del modelo nodal ya validado.
- La combinacion da la estimacion final:
$$V_{juntura}(f) = V_{salida\ filtro\ ambiente}(f)\cdot H_{cry}(f)$$

Notar que **no hace falta multiplicar por $H_{ambiente}$ simulado en ningun lado** -- eso es exactamente lo que se reemplaza por la medicion real. $H_{ambiente}$ medido sirve, aparte, para caracterizar el filtro ambiente en si mismo (que es lo que pide el item 3 del plan: analizar las etapas de filtrado, en conjunto y por separado) y como chequeo de consistencia contra el valor nominal.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter

try:
    from IPython.display import display
except ImportError:
    display = print

In [ ]:
# Copiado de filtros_expLT.ipynb (ya validado ahi contra los valores de
# la tesis de Fran) -- resuelve un circuito lineal por analisis nodal
# con una fuente de tension ideal (amplitud 1) entre dos terminales.
def nodal_solve_Vfuente(nodes, stampsY, terminales_fuente, ref, w):
    """'ref' tiene que ser la tierra real del circuito (la caja de
    aluminio del filtro ambiente), no un nodo cualquiera."""
    nodes2 = nodes + ['Ivs']
    idx = {n: i for i, n in enumerate(nodes2)}
    n = len(nodes2)
    Y = np.zeros((n, n), dtype=complex)
    b = np.zeros(n, dtype=complex)
    for a, c, y in stampsY:
        Y[idx[a], idx[a]] += y; Y[idx[a], idx[c]] -= y
        Y[idx[c], idx[c]] += y; Y[idx[c], idx[a]] -= y
    vp, vn = terminales_fuente
    Y[idx[vp], idx['Ivs']] += 1; Y[idx[vn], idx['Ivs']] -= 1
    Y[idx['Ivs'], idx[vp]] += 1; Y[idx['Ivs'], idx[vn]] -= 1
    b[idx['Ivs']] = 1.0
    r = idx[ref]
    keep = [i for i in range(n) if i != r]
    vr = np.linalg.solve(Y[np.ix_(keep, keep)], b[keep])
    v = np.zeros(n, dtype=complex)
    for j, i in enumerate(keep):
        v[i] = vr[j]
    return dict(zip(nodes2, v))


def _fmt_ing(x, pos):
    if x <= 0:
        return "0"
    for potencia, sufijo in [(1e9, "G"), (1e6, "M"), (1e3, "k")]:
        if x >= potencia * 0.999:
            return f"{x/potencia:g}{sufijo}"
    return f"{x:g}"


def _subs_segun_rango(lo, hi):
    decadas = np.log10(hi / lo) if lo > 0 else 8
    if decadas > 4:
        return (1.0,)
    elif decadas > 1.5:
        return (1.0, 2.0, 5.0)
    else:
        return (1.0, 2.0, 3.0, 5.0, 7.0)


def ejes_legibles(ax, x=True, y=True):
    """Ejes log con numeros normales (1, 10, 100, 1k, 1M...) en vez de
    notacion 10^x."""
    if x:
        subs = _subs_segun_rango(*ax.get_xlim())
        ax.xaxis.set_major_locator(LogLocator(base=10.0, subs=subs))
        ax.xaxis.set_major_formatter(FuncFormatter(_fmt_ing))
        ax.xaxis.set_minor_formatter(NullFormatter())
    if y:
        subs = _subs_segun_rango(*ax.get_ylim())
        ax.yaxis.set_major_locator(LogLocator(base=10.0, subs=subs))
        ax.yaxis.set_major_formatter(FuncFormatter(_fmt_ing))
        ax.yaxis.set_minor_formatter(NullFormatter())

In [ ]:
# Tabla 5.1 de la tesis de Fran: seis canales del filtro ambiente
CANALES = pd.DataFrame({
    "canal": [1, 2, 3, 4, 5, 6],
    "R_ta_ohm": [150.0, 150.0, 51.0, 51.0, 15.0, 15.0],
    "C_ta_uF": [2.2, 1.0, 2.2, 1.0, 2.2, 1.0],
    "fc_tesis_Hz": [180.0, 396.0, 530.0, 1165.0, 1801.0, 3962.0],
})

R_cry, C_cry = 22.0, 47e-9        # filtro criogenico, todos los canales iguales
R_N, R_sg = 500.0, 50e3           # resistencia normal / subgap de la juntura
C_J = 50e-12                      # capacidad aproximada de la juntura

display(CANALES)

## Modelo nodal de las cuatro lineas reales

Se reusa (generalizada) la Seccion 4 de `filtros_expLT.ipynb`: las cuatro lineas reales (corriente + sensado), cada una con sus dos etapas de filtro, convergiendo en los dos terminales fisicos de la juntura ($jp$, $jn$). La generalizacion permite elegir por cual linea entra la interferencia (`linea_fuente='sensado'` -- la del voltimetro, que es la que a vos te importa -- o `'corriente'`), y ademas devuelve *todos* los nodos internos, no solo $V(jp)$: eso permite leer por separado la tension en el nodo que esta justo entre el filtro ambiente y el criogenico -- exactamente el punto que vas a medir como "salida del filtro ambiente".

In [ ]:
def _nodos_linea(pref):
    """Los 10 nodos internos de una linea: extremo cercano, las dos
    etapas de filtro (ambiente y cry) en la rama '+' y en la '-'.
    'a2'/'b2' es el nodo justo entre el filtro ambiente y el cry -- el
    punto fisico donde el cable entra al criostato."""
    return {
        'near_p': f'{pref}_p', 'a1': f'{pref}_a1', 'a2': f'{pref}_a2', 'ca1': f'{pref}_ca1', 'ca2': f'{pref}_ca2',
        'near_n': f'{pref}_n', 'b1': f'{pref}_b1', 'b2': f'{pref}_b2', 'cb1': f'{pref}_cb1', 'cb2': f'{pref}_cb2',
    }


def _armar_dos_lineas(R_ta, C_ta, R_cry, C_cry, R_load, C_load, w, pref_activa, pref_pasiva):
    """Dos lineas identicas (corriente y sensado). La 'activa' lleva la
    fuente de tension (amplitud 1) entre sus dos extremos cercanos; la
    'pasiva' no tiene fuente propia (solo un resistor de 1 GOhm entre
    sus extremos, para que la matriz no quede singular con ese tramo
    flotando), pero sigue afectando jp/jn por sus propios capacitores
    criogenicos -- son los mismos filtros fisicos en las cuatro lineas."""
    L1 = _nodos_linea(pref_activa)
    L2 = _nodos_linea(pref_pasiva)
    nodes = ['GND', 'jp', 'jn'] + list(L1.values()) + list(L2.values())
    Yload = 1.0 / R_load + 1j * w * C_load
    stampsY = []
    for L in (L1, L2):
        stampsY += [
            (L['near_p'], L['a1'], 1.0 / R_ta), (L['a1'], L['a2'], 1.0 / R_ta),
            (L['near_n'], L['b1'], 1.0 / R_ta), (L['b1'], L['b2'], 1.0 / R_ta),
            (L['a1'], 'GND', 1j * w * C_ta), (L['a2'], 'GND', 1j * w * C_ta),
            (L['b1'], 'GND', 1j * w * C_ta), (L['b2'], 'GND', 1j * w * C_ta),
            (L['a2'], L['ca1'], 1.0 / R_cry), (L['ca1'], L['ca2'], 1.0 / R_cry), (L['ca2'], 'jp', 1.0 / R_cry),
            (L['b2'], L['cb1'], 1.0 / R_cry), (L['cb1'], L['cb2'], 1.0 / R_cry), (L['cb2'], 'jn', 1.0 / R_cry),
            (L['ca1'], L['cb1'], 1j * w * C_cry), (L['ca2'], L['cb2'], 1j * w * C_cry),
        ]
    stampsY.append(('jp', 'jn', Yload))
    stampsY.append((L2['near_p'], L2['near_n'], 1.0 / 1e9))
    return nodes, stampsY, L1['near_p'], L1['near_n']


def _resolver(f_hz, R_ta, C_ta, R_load, linea_fuente, R_cry, C_cry, C_load):
    """Resuelve el circuito completo y devuelve el diccionario de todos
    los nodos (con V_in=1) junto con el prefijo de la linea activa."""
    w = 2.0 * np.pi * f_hz
    pref_activa, pref_pasiva = ('S', 'I') if linea_fuente == 'sensado' else ('I', 'S')
    nodes, stampsY, vp, vn = _armar_dos_lineas(R_ta, C_ta, R_cry, C_cry, R_load, C_load, w, pref_activa, pref_pasiva)
    v = nodal_solve_Vfuente(nodes, stampsY, (vp, vn), ref='GND', w=w)
    return v, pref_activa


def ganancia_juntura(f_hz, R_ta, C_ta, R_load, linea_fuente='sensado', R_cry=R_cry, C_cry=C_cry, C_load=C_J):
    """H(f) = V(jp)/V_in -- ganancia de la cadena COMPLETA (ambiente +
    cry), igual que cuatro_lineas() de filtros_expLT.ipynb."""
    v, pref = _resolver(f_hz, R_ta, C_ta, R_load, linea_fuente, R_cry, C_cry, C_load)
    return v['jp']


def H_ambiente(f_hz, R_ta, C_ta, R_load, linea_fuente='sensado', R_cry=R_cry, C_cry=C_cry, C_load=C_J):
    """Ganancia SIMULADA del filtro ambiente solo (entrada -> nodo
    justo antes del criogenico). Se usa unicamente como referencia
    nominal para comparar contra la medicion real -- no hace falta
    para llegar a la juntura, porque ese tramo se mide."""
    v, pref = _resolver(f_hz, R_ta, C_ta, R_load, linea_fuente, R_cry, C_cry, C_load)
    return v[f'{pref}_a2']


def H_cry_efectivo(f_hz, R_ta, C_ta, R_load, linea_fuente='sensado', R_cry=R_cry, C_cry=C_cry, C_load=C_J):
    """Ganancia SIMULADA del tramo que va desde la salida del filtro
    ambiente (el nodo que se puede medir) hasta la juntura -- el UNICO
    tramo que sigue dependiendo del modelo, porque es inaccesible."""
    v, pref = _resolver(f_hz, R_ta, C_ta, R_load, linea_fuente, R_cry, C_cry, C_load)
    return v['jp'] / v[f'{pref}_a2']

### Validacion

Dos chequeos antes de usar esto: (1) que la generalizacion coincida con `cuatro_lineas` de `filtros_expLT.ipynb` cuando la fuente esta en la linea de corriente (que ya estaba validada ahi contra la tesis), y (2) que $H_{ambiente}\cdot H_{cry\ efectivo} = $ ganancia total, por construccion.

In [ ]:
def _cuatro_lineas_original(f_hz, R_ta, C_ta, R_cry, C_cry, R_load, C_load):
    """Copia exacta de cuatro_lineas() en filtros_expLT.ipynb (celda 27),
    fuente siempre en la linea de corriente. Solo para validar."""
    w = 2.0 * np.pi * f_hz
    nodes = ['GND', 'Vin_p', 'Ta1', 'Ta2', 'Ca1', 'Ca2', 'jp',
              'Vin_n', 'Tb1', 'Tb2', 'Cb1', 'Cb2', 'jn',
              'Sfar_a', 'Sa1', 'Sa2', 'Sca1', 'Sca2', 'Sfar_b', 'Sb1', 'Sb2', 'Scb1', 'Scb2']
    Yload = 1.0 / R_load + 1j * w * C_load
    stampsY = [
        ('Vin_p', 'Ta1', 1.0 / R_ta), ('Ta1', 'Ta2', 1.0 / R_ta),
        ('Vin_n', 'Tb1', 1.0 / R_ta), ('Tb1', 'Tb2', 1.0 / R_ta),
        ('Ta1', 'GND', 1j * w * C_ta), ('Ta2', 'GND', 1j * w * C_ta),
        ('Tb1', 'GND', 1j * w * C_ta), ('Tb2', 'GND', 1j * w * C_ta),
        ('Ta2', 'Ca1', 1.0 / R_cry), ('Ca1', 'Ca2', 1.0 / R_cry), ('Ca2', 'jp', 1.0 / R_cry),
        ('Tb2', 'Cb1', 1.0 / R_cry), ('Cb1', 'Cb2', 1.0 / R_cry), ('Cb2', 'jn', 1.0 / R_cry),
        ('Ca1', 'Cb1', 1j * w * C_cry), ('Ca2', 'Cb2', 1j * w * C_cry),
        ('jp', 'jn', Yload),
        ('Sfar_a', 'Sa1', 1.0 / R_ta), ('Sfar_b', 'Sb1', 1.0 / R_ta),
        ('Sfar_a', 'Sfar_b', 1.0 / 1e9),
        ('Sa1', 'Sa2', 1.0 / R_ta), ('Sb1', 'Sb2', 1.0 / R_ta),
        ('Sa1', 'GND', 1j * w * C_ta), ('Sa2', 'GND', 1j * w * C_ta),
        ('Sb1', 'GND', 1j * w * C_ta), ('Sb2', 'GND', 1j * w * C_ta),
        ('Sa2', 'Sca1', 1.0 / R_cry), ('Sca1', 'Sca2', 1.0 / R_cry), ('Sca2', 'jp', 1.0 / R_cry),
        ('Sb2', 'Scb1', 1.0 / R_cry), ('Scb1', 'Scb2', 1.0 / R_cry), ('Scb2', 'jn', 1.0 / R_cry),
        ('Sca1', 'Scb1', 1j * w * C_cry), ('Sca2', 'Scb2', 1j * w * C_cry),
    ]
    v = nodal_solve_Vfuente(nodes, stampsY, ('Vin_p', 'Vin_n'), ref='GND', w=w)
    return v['jp']


frecuencias_val = np.logspace(0, 6, 40)
diffs_original = [
    abs(_cuatro_lineas_original(f, 150.0, 2.2e-6, R_cry, C_cry, Rl, C_J)
        - ganancia_juntura(f, 150.0, 2.2e-6, Rl, linea_fuente='corriente'))
    for f in frecuencias_val for Rl in (R_N, R_sg)
]
diffs_identidad = [
    abs(ganancia_juntura(f, 150.0, 2.2e-6, Rl)
        - H_ambiente(f, 150.0, 2.2e-6, Rl) * H_cry_efectivo(f, 150.0, 2.2e-6, Rl))
    for f in frecuencias_val for Rl in (R_N, R_sg)
]
print(f"(1) diferencia maxima vs. cuatro_lineas original:      {max(diffs_original):.2e}  (esperado: ~0)")
print(f"(2) diferencia maxima H_ambiente*H_cry_efectivo:        {max(diffs_identidad):.2e}  (esperado: ~0, identidad)")
assert max(diffs_original) < 1e-8 and max(diffs_identidad) < 1e-8, "revisar el modelo"

### ¿Importa por cual linea entra la interferencia?

Igual que en la Seccion 4 de `filtros_expLT.ipynb`: la linea de corriente y la de sensado de un mismo canal tienen el mismo filtro (mismos $R_{ta}$, $C_{ta}$, $R_{cry}$, $C_{cry}$), asi que por reciprocidad de la red pasiva no importa cual de las dos "empuja" la senal -- da exactamente igual. Esto tambien vale para $H_{cry\ efectivo}$ sola, que es la que realmente se va a usar.

In [ ]:
diffs_reciprocidad = [
    abs(H_cry_efectivo(f, 150.0, 2.2e-6, R_N, linea_fuente='corriente')
        - H_cry_efectivo(f, 150.0, 2.2e-6, R_N, linea_fuente='sensado'))
    for f in frecuencias_val
]
print(f"diferencia maxima H_cry_efectivo, corriente vs. sensado: {max(diffs_reciprocidad):.2e}  (esperado: 0)")

## La curva que realmente hace falta: $H_{cry\ efectivo}(f)$, por canal

Esta es la unica pieza que sigue viniendo del modelo (no se puede medir). El resto -- la ganancia del filtro ambiente -- se reemplaza por la medicion real.

In [ ]:
frecuencias = np.logspace(0, 6, 400)

fig, axs = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, (nombre, R_load) in zip(axs, [("$R_N$ (normal)", R_N), ("$R_{sg}$ (subgap)", R_sg)]):
    for fila in CANALES.itertuples():
        R_ta, C_ta = fila.R_ta_ohm, fila.C_ta_uF * 1e-6
        h_db = 20 * np.log10([abs(H_cry_efectivo(f, R_ta, C_ta, R_load)) for f in frecuencias])
        ax.semilogx(frecuencias, h_db, label=f"canal {fila.canal}: {R_ta:.0f}$\\Omega$/{fila.C_ta_uF:.1f}$\\mu$F")
    ax.axvline(35700.0, color="gray", linestyle=":", linewidth=1, label="$f_c$ cry (tesis) = 35,7 kHz")
    ax.set_xlabel("Frecuencia [Hz]")
    ax.set_title(f"Carga: {nombre}")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, which="both")
    ejes_legibles(ax, y=False)
axs[0].set_ylabel(r"$H_{cry\ efectivo}(f) = |V(jp)/V_{salida\ ambiente}|$ [dB]")
fig.suptitle("Ganancia del tramo criogenico solo (el que hay que simular), por canal")
plt.tight_layout()
plt.show()

## De dBm del analizador a V, y el cuidado con la carga de 50 $\Omega$

Un analizador de espectro tipico entrega la amplitud en **dBm sobre 50 $\Omega$**, no en volts directamente:

$$V_{rms} = \sqrt{P\,[\mathrm{W}]\cdot Z}\,,\qquad P\,[\mathrm{W}] = 10^{(P_{dBm}/10)}\times 10^{-3}$$

**Cuidado antes de medir:** $R_{ta}$ llega a ser tan chico como $15\,\Omega$ -- comparable a los $50\,\Omega$ de entrada del analizador. Conectarlo directamente (sin sonda de alta impedancia) carga el nodo y puede modificar la tension real que hay ahi, tanto en la entrada como en la salida del filtro ambiente. Conviene usar una sonda de alta impedancia (activa/FET, o al menos $\times10$) en los dos puntos donde midas.

In [ ]:
def dbm_a_vrms(P_dBm, Z=50.0):
    P_watts = 10.0 ** (np.asarray(P_dBm) / 10.0) * 1e-3
    return np.sqrt(P_watts * Z)


def vrms_a_dbm(V_rms, Z=50.0):
    P_watts = np.asarray(V_rms) ** 2 / Z
    return 10.0 * np.log10(P_watts / 1e-3)


def H_ambiente_medido(V_entrada_medido, V_salida_medido):
    """H_ambiente(f) a partir de las dos mediciones reales (mismo array
    de frecuencias para las dos). Sirve para caracterizar el filtro
    ambiente en si mismo (item 3 del plan) y para compararlo con el
    valor nominal H_ambiente(...) simulado, como control de calidad."""
    return np.asarray(V_salida_medido) / np.asarray(V_entrada_medido)


def voltaje_estimado_en_juntura(frecuencias_hz, V_salida_ambiente_medido, R_ta, C_ta, R_load, linea_fuente='sensado'):
    """V_juntura(f) = V_salida_ambiente_medido(f) * |H_cry_efectivo(f)|.
    Ya NO hace falta la entrada del filtro ni H_ambiente simulado --
    ese tramo se reemplazo por la medicion real de la salida."""
    H = np.array([abs(H_cry_efectivo(f, R_ta, C_ta, R_load, linea_fuente=linea_fuente)) for f in frecuencias_hz])
    return np.asarray(V_salida_ambiente_medido) * H

## Ejemplo con datos de juguete

Todavia no hay datos reales. Este ejemplo simula las **dos** mediciones (entrada y salida del filtro ambiente, canal 1) a partir de un espectro de entrada inventado -- piso de ruido mas dos picos, en 50 Hz y en 2 kHz -- para mostrar el pipeline completo funcionando de punta a punta. Para usarlo con datos reales: reemplazar `V_entrada_ejemplo` y `V_salida_ejemplo` por las dos exportaciones del analizador (frecuencia + dBm, convertido con `dbm_a_vrms`), y `R_ta`/`C_ta`/`R_load` por el canal y el estado real de la juntura.

In [ ]:
frecuencias_ejemplo = np.logspace(0, 5, 300)
canal_medido = CANALES.iloc[0]      # canal 1: 150 ohm / 2.2 uF
R_load_medido = R_N                 # o R_sg, segun el estado de la juntura durante la medicion

# --- entrada: espectro de juguete, piso de ruido + dos picos (reemplazar por datos reales) ---
piso_dBm = -60.0
pico1 = 25.0 * np.exp(-0.5 * ((np.log10(frecuencias_ejemplo) - np.log10(50)) / 0.03) ** 2)
pico2 = 15.0 * np.exp(-0.5 * ((np.log10(frecuencias_ejemplo) - np.log10(2000)) / 0.05) ** 2)
V_entrada_ejemplo = dbm_a_vrms(piso_dBm + pico1 + pico2)

# --- salida: en un caso real es OTRA medicion independiente; aca se "fabrica"
# aplicando el H_ambiente simulado a la entrada, solo para que el ejemplo sea
# autoconsistente (en la realidad no va a coincidir exactamente con el
# nominal, por las tolerancias de los componentes reales) ---
H_amb_nominal = np.array([H_ambiente(f, canal_medido.R_ta_ohm, canal_medido.C_ta_uF * 1e-6, R_load_medido) for f in frecuencias_ejemplo])
V_salida_ejemplo = V_entrada_ejemplo * np.abs(H_amb_nominal)

# --- 1) caracterizar el filtro ambiente real, comparando contra el nominal ---
H_amb_medido = H_ambiente_medido(V_entrada_ejemplo, V_salida_ejemplo)

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(frecuencias_ejemplo, 20 * np.log10(H_amb_medido), label="$H_{ambiente}$ medido (real)", color="C2", linewidth=2)
ax.semilogx(frecuencias_ejemplo, 20 * np.log10(np.abs(H_amb_nominal)), label="$H_{ambiente}$ nominal (simulado)", color="gray", linestyle="--")
ax.set_xlabel("Frecuencia [Hz]")
ax.set_ylabel("Ganancia [dB]")
ax.set_title(f"Filtro ambiente canal {canal_medido.canal:.0f}: medido vs. nominal")
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which="both")
ejes_legibles(ax, y=False)
plt.tight_layout()
plt.show()

# --- 2) estimar lo que llega a la juntura, a partir de la salida MEDIDA ---
V_juntura_estimado = voltaje_estimado_en_juntura(
    frecuencias_ejemplo, V_salida_ejemplo,
    canal_medido.R_ta_ohm, canal_medido.C_ta_uF * 1e-6, R_load_medido,
)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.semilogx(frecuencias_ejemplo, vrms_a_dbm(V_entrada_ejemplo), label="medido: entrada del filtro ambiente", color="C0")
ax.semilogx(frecuencias_ejemplo, vrms_a_dbm(V_salida_ejemplo), label="medido: salida del filtro ambiente", color="C1")
ax.semilogx(frecuencias_ejemplo, vrms_a_dbm(V_juntura_estimado), label="estimado: en la juntura", color="C3")
ax.set_xlabel("Frecuencia [Hz]")
ax.set_ylabel("Amplitud [dBm, ref. 50 $\\Omega$]")
ax.set_title(f"Ejemplo con datos de juguete -- canal {canal_medido.canal:.0f}, carga $R_N$")
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which="both")
ejes_legibles(ax, y=False)
plt.tight_layout()
plt.show()

## ¿Cuánto cambia según dónde midas: bordes del voltímetro vs. salida del filtro ambiente?

Comparación directa: $H_{completo}(f)$ es la ganancia desde el voltímetro (punto A) hasta la juntura -- toda la cadena, ambiente + cry. $H_{cry\ efectivo}(f)$ es la ganancia desde la salida del filtro ambiente (punto B) hasta la juntura -- solo el tramo criogénico. La brecha entre las dos curvas es, por construcción, exactamente cuánto atenúa el filtro ambiente por sí solo ($H_{completo} = H_{ambiente}\cdot H_{cry\ efectivo}$, la identidad de la Sección de validación) -- y **no** es chica.

In [ ]:
frecuencias_gap = np.logspace(0, 5, 400)
R_load_ej = R_N

fig, axs = plt.subplots(1, 2, figsize=(13, 5))

canal1 = CANALES.iloc[0]
hA = 20 * np.log10([abs(ganancia_juntura(f, canal1.R_ta_ohm, canal1.C_ta_uF * 1e-6, R_load_ej)) for f in frecuencias_gap])
hB = 20 * np.log10([abs(H_cry_efectivo(f, canal1.R_ta_ohm, canal1.C_ta_uF * 1e-6, R_load_ej)) for f in frecuencias_gap])
axs[0].semilogx(frecuencias_gap, hA, label="desde A (voltímetro): cadena completa", color="C3")
axs[0].semilogx(frecuencias_gap, hB, label="desde B (salida ambiente): solo cry", color="C0")
axs[0].fill_between(frecuencias_gap, hA, hB, color="gray", alpha=0.15, label="brecha = atenuación del filtro ambiente")
axs[0].set_xlabel("Frecuencia [Hz]")
axs[0].set_ylabel("Ganancia [dB]")
axs[0].set_title(f"Canal {canal1.canal:.0f} ({canal1.R_ta_ohm:.0f}$\\Omega$/{canal1.C_ta_uF:.1f}$\\mu$F), carga $R_N$")
axs[0].legend(fontsize=8)
axs[0].grid(alpha=0.3, which="both")
ejes_legibles(axs[0], y=False)

for fila in CANALES.itertuples():
    R_ta, C_ta = fila.R_ta_ohm, fila.C_ta_uF * 1e-6
    brecha_db = np.array([
        20 * np.log10(abs(H_cry_efectivo(f, R_ta, C_ta, R_load_ej)) / abs(ganancia_juntura(f, R_ta, C_ta, R_load_ej)))
        for f in frecuencias_gap
    ])
    axs[1].semilogx(frecuencias_gap, brecha_db, label=f"canal {fila.canal}")
axs[1].set_xlabel("Frecuencia [Hz]")
axs[1].set_ylabel("Brecha entre A y B [dB]")
axs[1].set_title("Cuánto más atenúa el filtro ambiente solo, por canal")
axs[1].legend(fontsize=8)
axs[1].grid(alpha=0.3, which="both")
ejes_legibles(axs[1], y=False)

plt.tight_layout()
plt.show()

print("Brecha entre medir en A y en B, canal 1, algunas frecuencias:")
for f in [1, 100, 1000, 1e4, 3.57e4, 1e5]:
    hA_f = abs(ganancia_juntura(f, canal1.R_ta_ohm, canal1.C_ta_uF * 1e-6, R_load_ej))
    hB_f = abs(H_cry_efectivo(f, canal1.R_ta_ohm, canal1.C_ta_uF * 1e-6, R_load_ej))
    print(f"  f={f:>9.0f} Hz:  desde A = {20*np.log10(hA_f):7.2f} dB   desde B = {20*np.log10(hB_f):7.2f} dB   brecha = {20*np.log10(hB_f/hA_f):6.2f} dB")

## Resumen

- Fran confirmo que la etapa ambiente completa (entrada y salida del filtro) es medible. Eso permite reemplazar la mitad de la cadena simulada por una medicion real.
- $H_{ambiente}(f)$: se **mide** (entrada vs. salida del filtro ambiente), no se simula. Sirve tambien para caracterizar el filtro en si mismo (item 3) y compararlo contra el valor nominal.
- $H_{cry\ efectivo}(f)$: es lo unico que sigue **simulado**, porque el tramo criogenico es inaccesible. Ya esta calculado por canal y por estado de la juntura ($R_N$/$R_{sg}$) en este notebook.
- La estimacion final es $V_{juntura}(f) = V_{salida\ filtro\ ambiente}(f)\cdot H_{cry\ efectivo}(f)$ -- solo un factor simulado, en vez de dos.
- No importa por cual linea (corriente o sensado) entra la interferencia: da exactamente igual, por reciprocidad.
- Sigue valiendo la limitacion de fondo: esto estima la interferencia **externa** que se propaga desde donde se mide. No captura ruido que se genere **adentro** del criostato (termico, acoplamiento interno) -- eso es el item 4 del plan, y se mide comparando configuraciones, no calculando con esta funcion de transferencia.
- Cuidado con la impedancia de entrada del analizador (50 $\Omega$) frente a $R_{ta}$ (hasta 15 $\Omega$): usar sonda de alta impedancia en los dos puntos de medicion.

**Proximo paso:** reemplazar `V_entrada_ejemplo` y `V_salida_ejemplo` por las dos mediciones reales del analizador de espectro (entrada y salida del filtro ambiente, mismo canal).